# BBC News — Topic Modelling Experiment

Compares **LDA**, **NMF**, **BERTopic**, **KATM+KeyBERT+spaCy**, **NeuralLDA**, and **ProdLDA**
on the SetFit/bbc-news corpus (2,225 docs, 5 categories: business, entertainment, politics, sport, tech).

All bag-of-words methods use the same stopword list for a fair comparison.

**Kernel:** Python (katm-venv)

In [ ]:
import time
import math
import numpy as np
from collections import Counter

import nltk
from datasets import load_dataset, concatenate_datasets
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

from katm import KATM
from katm.octis_data_prep import load_dataset as load_octis_dataset

for _r in ["corpora/stopwords", "tokenizers/punkt", "tokenizers/punkt_tab"]:
    try:
        nltk.data.find(_r)
    except LookupError:
        nltk.download(_r.split("/")[1], quiet=True)

## Parameters

In [ ]:
# ── Shared ────────────────────────────────────────────────────────────────────
N_TOPICS    = 5
TOP_N_WORDS = 10

# ── KATM ─────────────────────────────────────────────────────────────────────
N_KEYPHRASES         = 10
KEYBERT_NGRAM_RANGE  = (1, 2)
MIN_ANCHOR_DF        = 3
MAX_ANCHOR_DF_RATIO  = 0.4
ANCHOR_DEDUP_THRESH  = 0.85
SOFT_THRESHOLD       = 0.15
MIN_DF               = 3
MMR_DIVERSITY        = 0.2
MMR_MAX_SIM          = 0.85
CONTENT_WORDS_METHOD = "spacy"

# ── LDA / NMF ────────────────────────────────────────────────────────────────
LDA_MAX_ITER = 100
LDA_MAX_DF   = 0.95
LDA_MIN_DF   = 3

# ── BERTopic ─────────────────────────────────────────────────────────────────
BERT_MIN_TOPIC_SIZE = 15

## Stopwords

In [ ]:
_EXTRA_STOP = {
    "edu", "com", "org", "net", "gov", "www", "http", "re", "cc", "writes",
    "article", "subject", "lines", "like", "just", "don", "use", "know",
    "think", "time", "want", "good", "make", "look", "said", "say", "way",
    "does", "did", "got", "ll", "ve", "haven", "isn", "aren", "wasn",
    "doesn", "didn", "wouldn", "couldn", "shouldn",
}
_ALL_STOP  = set(nltk.corpus.stopwords.words("english")) | _EXTRA_STOP
_STOP_LIST = sorted(_ALL_STOP)
print(f"Stopword list size: {len(_ALL_STOP)}")

## Data — SetFit/bbc-news (train + test combined)

In [ ]:
ds_train   = load_dataset("SetFit/bbc-news", split="train")
ds_test    = load_dataset("SetFit/bbc-news", split="test")
ds         = concatenate_datasets([ds_train, ds_test])

documents  = [r["text"] for r in ds]
label_text = [r["label_text"] for r in ds]
names      = sorted(set(label_text))
name2id    = {n: i for i, n in enumerate(names)}
labels     = np.array([name2id[t] for t in label_text])

print(f"Docs      : {len(documents)}")
print(f"Categories: {names}")
print(f"Avg length: {np.mean([len(d) for d in documents]):.0f} chars")
for name in names:
    print(f"  {name}: {label_text.count(name)} docs")

## Evaluation helpers

In [ ]:
_embed_model = SentenceTransformer("all-mpnet-base-v2")

def purity(assignments, labels):
    total, score = len(assignments), 0
    for tid in set(assignments):
        idxs = [i for i, a in enumerate(assignments) if a == tid]
        score += Counter(labels[i] for i in idxs).most_common(1)[0][1]
    return score / total

def semantic_coherence(topic_words_list):
    scores = []
    for words in topic_words_list:
        if len(words) < 2:
            continue
        embs = _embed_model.encode(words, show_progress_bar=False)
        n    = len(embs)
        sims = [float(cosine_similarity([embs[i]], [embs[j]])[0][0])
                for i in range(n) for j in range(i+1, n)]
        scores.append(np.mean(sims))
    return float(np.mean(scores)) if scores else 0.0

def topic_diversity(topic_words_list):
    all_words = [w for words in topic_words_list for w in words]
    return len(set(all_words)) / len(all_words) if all_words else 0.0

def eval_metrics(assignments, labels, topic_words_list):
    pur = purity(list(assignments), list(labels))
    nmi = normalized_mutual_info_score(labels, assignments)
    ari = adjusted_rand_score(labels, assignments)
    coh = semantic_coherence(topic_words_list)
    div = topic_diversity(topic_words_list)
    return pur, nmi, ari, coh, div

def alignment_table(assignments, topic_words_list, n_topics):
    print(f"  {'T':<3}  {'Top words':<70}  dominant category")
    print("-" * 100)
    for t in range(n_topics):
        words = topic_words_list[t] if t < len(topic_words_list) else []
        idxs  = [i for i, a in enumerate(assignments) if a == t]
        if not idxs:
            print(f"T{t:2d}  (empty)")
            continue
        dom_cat, dom_cnt = Counter(labels[i] for i in idxs).most_common(1)[0]
        pct = dom_cnt / len(idxs) * 100
        print(f"T{t:2d}  {', '.join(words):<70}  {names[dom_cat]} ({pct:.0f}%)")

results = {}

## LDA

In [ ]:
t0  = time.time()
vec = CountVectorizer(
    max_df=LDA_MAX_DF, min_df=LDA_MIN_DF,
    stop_words=_STOP_LIST,
    token_pattern=r"(?u)\b[a-zA-Z]{4,}\b",
)
dtm   = vec.fit_transform(documents)
vocab = vec.get_feature_names_out()

lda = LatentDirichletAllocation(
    n_components=N_TOPICS, max_iter=LDA_MAX_ITER,
    random_state=42, n_jobs=-1,
)
lda.fit(dtm)
lda_assign = lda.transform(dtm).argmax(axis=1)
lda_time   = time.time() - t0

lda_words = [[vocab[i] for i in comp.argsort()[:-TOP_N_WORDS-1:-1]] for comp in lda.components_]
pur, nmi, ari, coh, div = eval_metrics(lda_assign, labels, lda_words)
results["LDA"] = dict(time=lda_time, coh=coh, div=div, purity=pur, nmi=nmi, ari=ari)

print(f"LDA  {lda_time:.1f}s  coh={coh:.4f}  div={div*100:.1f}%  purity={pur:.3f}  NMI={nmi:.3f}  ARI={ari:.3f}")
print()
alignment_table(lda_assign, lda_words, N_TOPICS)

## NMF (Non-negative Matrix Factorisation)

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

t0 = time.time()
_tfidf_tr = TfidfTransformer()
_dtm_tf   = _tfidf_tr.fit_transform(dtm)
nmf = NMF(n_components=N_TOPICS, random_state=42, max_iter=400, init="nndsvda")
_H_nmf     = nmf.fit_transform(_dtm_tf)
nmf_assign = _H_nmf.argmax(axis=1)
nmf_time   = time.time() - t0

nmf_words = [[vocab[i] for i in comp.argsort()[:-TOP_N_WORDS-1:-1]] for comp in nmf.components_]
pur, nmi_s, ari, coh, div = eval_metrics(nmf_assign, labels, nmf_words)
results["NMF"] = dict(time=nmf_time, coh=coh, div=div, purity=pur, nmi=nmi_s, ari=ari)

print(f"NMF  {nmf_time:.1f}s  coh={coh:.4f}  div={div*100:.1f}%  purity={pur:.3f}  NMI={nmi_s:.3f}  ARI={ari:.3f}")
print()
alignment_table(nmf_assign, nmf_words, N_TOPICS)

## BERTopic

In [ ]:
t0     = time.time()
bt_vec = CountVectorizer(stop_words=_STOP_LIST, token_pattern=r"(?u)\b[a-zA-Z]{4,}\b")
bt     = BERTopic(nr_topics=N_TOPICS, min_topic_size=BERT_MIN_TOPIC_SIZE,
                  vectorizer_model=bt_vec, verbose=False)
bt_raw, _ = bt.fit_transform(documents)
bt_raw    = np.array(bt_raw)
bt_time   = time.time() - t0

topic_ids = sorted(t for t in set(bt_raw) if t != -1)
id_remap  = {old: new for new, old in enumerate(topic_ids)}
bt_assign = np.array([id_remap.get(a, len(topic_ids)) for a in bt_raw])
bt_words  = [[w for w, _ in bt.get_topic(tid)[:TOP_N_WORDS]] for tid in topic_ids]
outliers  = int((bt_raw == -1).sum())

pur, nmi, ari, coh, div = eval_metrics(bt_assign, labels, bt_words)
results["BERTopic"] = dict(time=bt_time, coh=coh, div=div, purity=pur, nmi=nmi, ari=ari,
                            n_topics=len(topic_ids), outliers=outliers)

print(f"BERTopic  {bt_time:.1f}s  coh={coh:.4f}  div={div*100:.1f}%  purity={pur:.3f}  "
      f"NMI={nmi:.3f}  ARI={ari:.3f}  ({len(topic_ids)} topics, {outliers} outliers)")
print()
alignment_table(bt_assign, bt_words, len(topic_ids))

## KATM + KeyBERT + spaCy

In [ ]:
t0 = time.time()
katm = KATM(
    kp_algorithm          = "keybert",
    n_keyphrases          = N_KEYPHRASES,
    keybert_ngram_range   = KEYBERT_NGRAM_RANGE,
    n_topics              = N_TOPICS,
    min_anchor_df         = MIN_ANCHOR_DF,
    max_anchor_df_ratio   = MAX_ANCHOR_DF_RATIO,
    anchor_dedup_threshold= ANCHOR_DEDUP_THRESH,
    soft_threshold        = SOFT_THRESHOLD,
    min_df                = MIN_DF,
    top_n_words           = TOP_N_WORDS,
    mmr_diversity         = MMR_DIVERSITY,
    mmr_max_sim           = MMR_MAX_SIM,
    content_words_method  = CONTENT_WORDS_METHOD,
)
katm.fit(documents)
katm_time = time.time() - t0

kb_assign = np.array([int(np.argmax(p)) for p in katm.doc_topic_probs_])
kb_words  = [[w for w, _ in katm.get_topic_words(t, n=TOP_N_WORDS)] for t in range(N_TOPICS)]

pur, nmi, ari, coh, div = eval_metrics(kb_assign, labels, kb_words)
results["KATM+spaCy"] = dict(time=katm_time, coh=coh, div=div, purity=pur, nmi=nmi, ari=ari)

print(f"KATM+spaCy  {katm_time:.1f}s  coh={coh:.4f}  div={div*100:.1f}%  "
      f"purity={pur:.3f}  NMI={nmi:.3f}  ARI={ari:.3f}")
print()
alignment_table(kb_assign, kb_words, N_TOPICS)

## NeuralLDA & ProdLDA (OCTIS / AVITM)

`NeuralLDA` and `ProdLDA` are variational autoencoder topic models from the OCTIS benchmark
library. They require a bag-of-words corpus and are trained on OCTIS's preprocessed BBC News
(spaCy lemmatisation, `min_df=5`). NMI, Purity, and ARI are computed on the training
partition (~85 % of corpus); semantic coherence and diversity use the same embedding model.

In [ ]:
_bbc_octis_ds        = load_octis_dataset("BBC_News")
_bbc_oc_train, _bbc_oc_val, _bbc_oc_test = _bbc_octis_ds.get_partitioned_corpus()
_bbc_last_train      = _bbc_octis_ds.get_metadata().get("last-training-doc", len(_bbc_oc_train))
_bbc_oc_labels_all   = _bbc_octis_ds.get_labels()
_bbc_oc_label_names  = sorted(set(_bbc_oc_labels_all))
_bbc_oc_label2id     = {l: i for i, l in enumerate(_bbc_oc_label_names)}
_bbc_train_labels    = np.array([_bbc_oc_label2id[l] for l in _bbc_oc_labels_all[:_bbc_last_train]])

print(f"OCTIS BBC_News: {len(_bbc_octis_ds.get_corpus())} docs, "
      f"vocab={len(_bbc_octis_ds.get_vocabulary())}")
print(f"Split: {len(_bbc_oc_train)} train / {len(_bbc_oc_val)} val / {len(_bbc_oc_test)} test")
print(f"Categories: {_bbc_oc_label_names}")

### NeuralLDA

In [ ]:
from octis.models.NeuralLDA import NeuralLDA as _OctisNeuralLDA

t0 = time.time()
try:
    _nlda_model  = _OctisNeuralLDA(num_topics=N_TOPICS, num_epochs=100, batch_size=64, lr=2e-3)
    _nlda_out    = _nlda_model.train_model(_bbc_octis_ds, top_words=TOP_N_WORDS)
    nlda_time    = time.time() - t0
    nlda_words   = _nlda_out["topics"]
    _nlda_tdm    = _nlda_out["topic-document-matrix"]
    _nlda_assign = np.argmax(_nlda_tdm, axis=0)

    _nmi_v = normalized_mutual_info_score(_bbc_train_labels, _nlda_assign)
    _ari_v = adjusted_rand_score(_bbc_train_labels, _nlda_assign)
    _pur_v = sum(
        Counter(_bbc_train_labels[_nlda_assign == k]).most_common(1)[0][1]
        for k in range(N_TOPICS) if (_nlda_assign == k).sum() > 0
    ) / len(_bbc_train_labels)
    _coh_v = semantic_coherence(nlda_words)
    _div_v = topic_diversity(nlda_words)

    results["NeuralLDA"] = dict(time=nlda_time, coh=_coh_v, div=_div_v,
                                 purity=_pur_v, nmi=_nmi_v, ari=_ari_v)
    print(f"NeuralLDA  {nlda_time:.1f}s  coh={_coh_v:.4f}  div={_div_v*100:.1f}%  "
          f"purity={_pur_v:.3f}  NMI={_nmi_v:.3f}  ARI={_ari_v:.3f}")
    print()
    print(f"  {'T':<3}  {'Top words':<70}  dominant category")
    print("-" * 100)
    for _t in range(len(nlda_words)):
        _words = nlda_words[_t]
        _idxs  = np.where(_nlda_assign == _t)[0]
        if len(_idxs) == 0:
            print(f"T{_t:2d}  (empty)"); continue
        _dom_id, _dom_cnt = Counter(_bbc_train_labels[_idxs]).most_common(1)[0]
        _pct = _dom_cnt / len(_idxs) * 100
        print(f"T{_t:2d}  {', '.join(_words):<70}  {_bbc_oc_label_names[_dom_id]} ({_pct:.0f}%)")
except Exception as _e:
    nlda_time = time.time() - t0
    results["NeuralLDA"] = dict(time=nlda_time, coh=float("nan"), div=float("nan"),
                                 purity=float("nan"), nmi=float("nan"), ari=float("nan"))
    print(f"NeuralLDA FAILED: {_e}")

### ProdLDA

In [ ]:
from octis.models.ProdLDA import ProdLDA as _OctisProdLDA

t0 = time.time()
try:
    _plda_model  = _OctisProdLDA(num_topics=N_TOPICS, num_epochs=100, batch_size=64, lr=2e-3)
    _plda_out    = _plda_model.train_model(_bbc_octis_ds, top_words=TOP_N_WORDS)
    plda_time    = time.time() - t0
    plda_words   = _plda_out["topics"]
    _plda_tdm    = _plda_out["topic-document-matrix"]
    _plda_assign = np.argmax(_plda_tdm, axis=0)

    _nmi_v2 = normalized_mutual_info_score(_bbc_train_labels, _plda_assign)
    _ari_v2 = adjusted_rand_score(_bbc_train_labels, _plda_assign)
    _pur_v2 = sum(
        Counter(_bbc_train_labels[_plda_assign == k]).most_common(1)[0][1]
        for k in range(N_TOPICS) if (_plda_assign == k).sum() > 0
    ) / len(_bbc_train_labels)
    _coh_v2 = semantic_coherence(plda_words)
    _div_v2 = topic_diversity(plda_words)

    results["ProdLDA"] = dict(time=plda_time, coh=_coh_v2, div=_div_v2,
                               purity=_pur_v2, nmi=_nmi_v2, ari=_ari_v2)
    print(f"ProdLDA  {plda_time:.1f}s  coh={_coh_v2:.4f}  div={_div_v2*100:.1f}%  "
          f"purity={_pur_v2:.3f}  NMI={_nmi_v2:.3f}  ARI={_ari_v2:.3f}")
    print()
    print(f"  {'T':<3}  {'Top words':<70}  dominant category")
    print("-" * 100)
    for _t in range(len(plda_words)):
        _words = plda_words[_t]
        _idxs  = np.where(_plda_assign == _t)[0]
        if len(_idxs) == 0:
            print(f"T{_t:2d}  (empty)"); continue
        _dom_id, _dom_cnt = Counter(_bbc_train_labels[_idxs]).most_common(1)[0]
        _pct = _dom_cnt / len(_idxs) * 100
        print(f"T{_t:2d}  {', '.join(_words):<70}  {_bbc_oc_label_names[_dom_id]} ({_pct:.0f}%)")
except Exception as _e:
    plda_time = time.time() - t0
    results["ProdLDA"] = dict(time=plda_time, coh=float("nan"), div=float("nan"),
                               purity=float("nan"), nmi=float("nan"), ari=float("nan"))
    print(f"ProdLDA FAILED: {_e}")

## Results comparison

In [ ]:
print(f"{'Method':<18} {'Time':>7}  {'SemCoh':>7}  {'Div%':>6}  {'Purity':>7}  {'NMI':>7}  {'ARI':>7}")
print("-" * 80)
for method, r in results.items():
    note = ""
    if "n_topics" in r:
        note = f"  ({r['n_topics']} topics, {r['outliers']} outliers)"
    elif method in ("NeuralLDA", "ProdLDA"):
        note = "  [OCTIS preprocessing *]"
    t_v   = r.get("time", 0.0)
    coh_v = r.get("coh", float("nan"))
    div_v = r.get("div", float("nan"))
    pur_v = r.get("purity", float("nan"))
    nmi_v = r.get("nmi", float("nan"))
    ari_v = r.get("ari", float("nan"))
    _fv = lambda v: f"{v:>7.4f}" if not math.isnan(v) else f"{'NaN':>7}"
    _dv = f"{div_v*100:>5.1f}%" if not math.isnan(div_v) else "  NaN%"
    print(f"{method:<18} {t_v:>6.1f}s  {_fv(coh_v)}  {_dv}  {_fv(pur_v)}  {_fv(nmi_v)}  {_fv(ari_v)}{note}")
print()
print("* NeuralLDA / ProdLDA: OCTIS corpus (spaCy lemmatisation, min_df=5);")
print("  NMI / Purity / ARI measured on training partition (~85% of corpus).")